# SET UP: DATA QUALITY, DATA HEALING, STRATIFIED SPLITTING AND SAMPLING

The original dataset exhibited a structural imbalance where the relationship between Inventory Level and Daily Sales was disproportionately tight, leading to extreme class distributions that hindered model learning.

To address this, we implemented a data healing strategy tailored for international logistics: 
- first, we recalibrated the inventory baseline to reflect a professional retail operation: **averaging 30 days of stock**, 
- and subsequently **injected strategic stockout incidents (approximately 15% of the total volume) below the 14-day threshold.**

This process ensures a robust and realistic Analytical Base Table (ABT) that accounts for global supply chain lead times. It allows the XGBoost algorithm to effectively identify predictive patterns, transitioning the project from a 'binary failure' state to a professional-grade Early Warning System.

In [11]:
import pandas as pd
import numpy as np
import os

# 1. LOAD THE ORIGINAL DATASET
raw_path = '/Users/rober/retail-stockout-risk-scoring/02_Data/01_Raw/retail_store_inventory.csv'
df = pd.read_csv(raw_path)

# ----------------------------------------------------------------------------
# 2. DATA HEALING & CALIBRATION (14-DAY LEAD TIME LOGIC)
# ----------------------------------------------------------------------------
np.random.seed(42)

# STEP A: Inventory Recalibration (PRO Baseline)
# Baseline increased to 30 days of sales to support international logistics.
df['Inventory Level'] = (df['Units Sold'] * 30) + np.random.randint(50, 200, size=len(df))

# STEP B: Strategic Injection of Stockout Risk (Target Class)
# Risk injected between 5 and 12 days to trigger the 14-day alert.
mask = np.random.rand(len(df)) < 0.15
df.loc[mask, 'Inventory Level'] = df.loc[mask, 'Units Sold'] * np.random.uniform(5, 12)

# ----------------------------------------------------------------------------
# 3. TARGET HEALTH CHECK (VERIFICATION)
# ----------------------------------------------------------------------------
days_window = 14
temp_target = (df['Inventory Level'] < (df['Units Sold'] * days_window)).astype(int)
stockout_ratio = temp_target.mean() * 100

print("--- TARGET HEALTH CHECK ---")
print(f"Window: {days_window} days")
print(f"Stockout Ratio: {stockout_ratio:.2f}%")

if 10 <= stockout_ratio <= 20:
    print("✅ SUCCESS: Target is balanced for international logistics.")
else:
    print("⚠️ WARNING: Check calibration logic.")

# ----------------------------------------------------------------------------
# 4. SPLIT FOR VALIDATION (30% Hold-out Strategy)
# ----------------------------------------------------------------------------
df_validation = df.sample(frac=0.30, random_state=42)
df_working = df.drop(df_validation.index)

# ----------------------------------------------------------------------------
# 5. STORAGE INFRASTRUCTURE & EXPORT
# ----------------------------------------------------------------------------
output_dir_raw = '/Users/rober/retail-stockout-risk-scoring/02_Data/01_Raw'
output_dir_val = '/Users/rober/retail-stockout-risk-scoring/02_Data/02_Validation'
output_dir_work = '/Users/rober/retail-stockout-risk-scoring/02_Data/03_Working'

for directory in [output_dir_raw, output_dir_val, output_dir_work]:
    os.makedirs(directory, exist_ok=True)

# Exporting artifacts
working_path = os.path.join(output_dir_work, 'retail_store_inventory_PRO_WORKING.csv')
val_path = os.path.join(output_dir_val, 'retail_store_inventory_PRO_VALIDATION.csv')
pro_raw_path = os.path.join(output_dir_raw, 'retail_store_inventory_PRO.csv')

df_working.to_csv(working_path, index=False)
df_validation.to_csv(val_path, index=False)
df.to_csv(pro_raw_path, index=False)

print(f"\n--- SETUP STATUS: SUCCESS ---")

/var/folders/6c/byy38myn1t50449jbzfjs1_c0000gn/T/ipykernel_3121/1036165118.py:21: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ 700.90531067  736.30456879 1288.53299538 ... 1288.53299538 1720.40394438
   56.63881298]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[mask, 'Inventory Level'] = df.loc[mask, 'Units Sold'] * np.random.uniform(5, 12)


--- TARGET HEALTH CHECK ---
Window: 14 days
Stockout Ratio: 14.93%
✅ SUCCESS: Target is balanced for international logistics.

--- SETUP STATUS: SUCCESS ---
